# Text-to-Image with Stable Diffusion (Diffusers)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/stable_diffusion_text_to_image.ipynb)

Diffusion models generate images by learning to reverse a noising process: start from pure noise, repeatedly denoise while conditioning on the text embedding.

Uses HuggingFace `diffusers` - fully free on Colab GPU (*Runtime > Change runtime type > T4*). First run downloads ~4 GB.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

## 1. Load the pipeline (fp16 for T4-friendly memory)

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None)          # keep default checker in production!
pipe = pipe.to("cuda")

## 2. Generate

In [ ]:
prompt = "a watercolor painting of a robot reading a book in a garden, soft light"
negative_prompt = "blurry, low quality, distorted"

image = pipe(prompt,
             negative_prompt=negative_prompt,
             num_inference_steps=30,        # more steps = finer detail, slower
             guidance_scale=7.5).images[0]  # how strictly to follow the prompt
image

## 3. Prompt anatomy that actually matters

In [ ]:
base = "portrait of an old fisherman"
styles = ["", "photorealistic, 85mm lens",
          "studio ghibli style", "van gogh oil painting"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
generator = torch.Generator("cuda").manual_seed(7)
for ax, s in zip(axes, styles):
    img = pipe(base + (", " + s if s else base), generator=generator,
               num_inference_steps=25).images[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(s or "bare prompt", fontsize=9)
plt.show()

## Key controls
| Knob | Effect |
|---|---|
| `num_inference_steps` | 20-50; higher = sharper, slower |
| `guidance_scale` | 5-9; higher obeys prompt harder, can burn contrast |
| `negative_prompt` | subtracts concepts (quality killers, artifacts) |
| `seed` | reproducibility |

Next-level tools: **img2img** (redraw existing picture), **ControlNet** (pose/edges control), **SDXL** (higher resolution).